# Phase 5 — Feature Engineering and Data Preparation

This phase focuses on transforming the cleaned dataset into a machine-learning-ready forecasting dataset.

The main objective of feature engineering is to create meaningful predictors that help machine learning models capture:

* seasonal patterns
* time-based behavior
* sales trends
* promotional effects
* store-level differences

In this phase, new time-based and forecasting-oriented features will be created to improve the predictive performance of the models.

The phase will also include:

* missing value treatment
* categorical encoding
* lag feature creation
* rolling window features
* feature transformation
* chronological train-test splitting

Special attention will be given to preventing data leakage since this project involves time-series forecasting.


In [1]:
# import data handling libraries
import pandas as pd
import numpy as np


# import visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

#

In [2]:
# Import dataset
sales_df = pd.read_csv('/home/emeka/projects/ML_Portfolio/retail-sales-forecasting/data/cleaned_sales_data.csv')

In [3]:
# Calculate percentage of missing values in each column
missing_percentages = sales_df.isnull().mean() * 100
missing_percentages

Store            0.000000
Dept             0.000000
Date             0.000000
Weekly_Sales     0.000000
IsHoliday        0.000000
Temperature      0.000000
Fuel_Price       0.000000
MarkDown1       64.257181
MarkDown2       73.611025
MarkDown3       67.480845
MarkDown4       67.984676
MarkDown5       64.079038
CPI              0.000000
Unemployment     0.000000
Type             0.000000
Size             0.000000
Month            0.000000
Year             0.000000
dtype: float64

In [4]:
# List markdown columns
markdown_columns = [
    'MarkDown1',
    'MarkDown2', 
    'MarkDown3',
    'MarkDown4',
    'MarkDown5'
]

# fill missing values in markdown columns with 0
sales_df[markdown_columns] = sales_df[markdown_columns].fillna(0)

# confirm that there are no more missing values in markdown columns
sales_df[markdown_columns].isnull().sum()

MarkDown1    0
MarkDown2    0
MarkDown3    0
MarkDown4    0
MarkDown5    0
dtype: int64

## Missing Value Treatment

Missing values were found only in the markdown columns (`MarkDown1` to `MarkDown5`). These columns represent promotional markdown activities.

The missing values were replaced with `0` because the null values likely indicate that no promotion or markdown campaign was active during those periods rather than missing data collection.

Other columns did not contain missing values, so no additional imputation was required.


In [5]:
# create time-based features from the 'Date' column
sales_df['Date'] = pd.to_datetime(sales_df['Date'])
sales_df['Quarter'] = sales_df['Date'].dt.quarter.astype(int)
sales_df['WeekOfYear'] = sales_df['Date'].dt.isocalendar().week.astype(int)

In [6]:
sales_df[['Date', 'Quarter', 'WeekOfYear', 'Month', 'Year']].head()

,Date,Quarter,WeekOfYear,Month,Year
0,2010-02-05,1,5,2,2010
1,2010-02-12,1,6,2,2010
2,2010-02-19,1,7,2,2010
3,2010-02-26,1,8,2,2010
4,2010-03-05,1,9,3,2010


## Time-Based Feature Engineering

New time-based features were created from the `Date` column to help the forecasting models capture seasonal and calendar-related sales patterns.

The following features were extracted:

* `Year`
* `Month`
* `Quarter`
* `WeekOfYear`

These features were created to help the models learn:

* yearly sales trends
* monthly seasonality
* quarterly business patterns
* recurring weekly sales behavior

Time-based features are important in forecasting because retail sales often change across different periods of the year.


In [7]:
# convert boolean holiday values to integers
sales_df['IsHoliday'] = sales_df['IsHoliday'].astype(int)

In [8]:
# one-hot encode store type
sales_df = pd.get_dummies(sales_df, columns=['Type'], drop_first=True, prefix='Type', prefix_sep='_', dtype=int)
sales_df.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Size,Month,Year,Quarter,WeekOfYear,Type_B,Type_C
0,1,1,2010-02-05,24924.50,0,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,8.106,151315,2,2010,1,5,0,0
1,1,1,2010-02-12,46039.49,1,38.51,2.548,0.0,0.0,0.0,0.0,0.0,211.242170,8.106,151315,2,2010,1,6,0,0
2,1,1,2010-02-19,41595.55,0,39.93,2.514,0.0,0.0,0.0,0.0,0.0,211.289143,8.106,151315,2,2010,1,7,0,0
3,1,1,2010-02-26,19403.54,0,46.63,2.561,0.0,0.0,0.0,0.0,0.0,211.319643,8.106,151315,2,2010,1,8,0,0
4,1,1,2010-03-05,21827.90,0,46.50,2.625,0.0,0.0,0.0,0.0,0.0,211.350143,8.106,151315,3,2010,1,9,0,0


In [9]:
sales_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 421570 entries, 0 to 421569
Data columns (total 21 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   Store         421570 non-null  int64         
 1   Dept          421570 non-null  int64         
 2   Date          421570 non-null  datetime64[ns]
 3   Weekly_Sales  421570 non-null  float64       
 4   IsHoliday     421570 non-null  int64         
 5   Temperature   421570 non-null  float64       
 6   Fuel_Price    421570 non-null  float64       
 7   MarkDown1     421570 non-null  float64       
 8   MarkDown2     421570 non-null  float64       
 9   MarkDown3     421570 non-null  float64       
 10  MarkDown4     421570 non-null  float64       
 11  MarkDown5     421570 non-null  float64       
 12  CPI           421570 non-null  float64       
 13  Unemployment  421570 non-null  float64       
 14  Size          421570 non-null  int64         
 15  Month         421

## Categorical Feature Encoding

Categorical variables were converted into numerical format to make them suitable for machine learning models.

The `IsHoliday` column was converted from boolean values to binary integers (`0` and `1`) because it contains only two possible categories.

The `Type` column was one-hot encoded to create separate store type indicators:

* `Type_B`
* `Type_C`

`drop_first=True` was used during encoding to avoid multicollinearity and reduce redundant features in the dataset.


In [10]:
# sort dataset by store, department, and date
sales_df = sales_df.sort_values(
    by=['Store', 'Dept', 'Date']
)

In [11]:
sales_df.head(10)

,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Size,Month,Year,Quarter,WeekOfYear,Type_B,Type_C
0,1,1,2010-02-05,24924.50,0,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,8.106,151315,2,2010,1,5,0,0
1,1,1,2010-02-12,46039.49,1,38.51,2.548,0.0,0.0,0.0,0.0,0.0,211.242170,8.106,151315,2,2010,1,6,0,0
2,1,1,2010-02-19,41595.55,0,39.93,2.514,0.0,0.0,0.0,0.0,0.0,211.289143,8.106,151315,2,2010,1,7,0,0
3,1,1,2010-02-26,19403.54,0,46.63,2.561,0.0,0.0,0.0,0.0,0.0,211.319643,8.106,151315,2,2010,1,8,0,0
4,1,1,2010-03-05,21827.90,0,46.50,2.625,0.0,0.0,0.0,0.0,0.0,211.350143,8.106,151315,3,2010,1,9,0,0
5,1,1,2010-03-12,21043.39,0,57.79,2.667,0.0,0.0,0.0,0.0,0.0,211.380643,8.106,151315,3,2010,1,10,0,0
6,1,1,2010-03-19,22136.64,0,54.58,2.720,0.0,0.0,0.0,0.0,0.0,211.215635,8.106,151315,3,2010,1,11,0,0
7,1,1,2010-03-26,26229.21,0,51.45,2.732,0.0,0.0,0.0,0.0,0.0,211.018042,8.106,151315,3,2010,1,12,0,0
8,1,1,2010-04-02,57258.43,0,62.27,2.719,0.0,0.0,0.0,0.0,0.0,210.820450,7.808,151315,4,2010,2,13,0,0
9,1,1,2010-04-09,42960.91,0,65.86,2.770,0.0,0.0,0.0,0.0,0.0,210.622857,7.808,151315,4,2010,2,14,0,0


In [12]:
# create lag features using previous sales values
sales_df['Lag_1'] = sales_df.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(1)
sales_df['Lag_4'] = sales_df.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(4)
sales_df['Lag_12'] = sales_df.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(12)

In [13]:
# inspect lag features
sales_df[
    [
        'Store',
        'Dept',
        'Date',
        'Weekly_Sales',
        'Lag_1',
        'Lag_4',
        'Lag_12'
    ]
].head(15)

,Store,Dept,Date,Weekly_Sales,Lag_1,Lag_4,Lag_12
0,1,1,2010-02-05,24924.50,NaN,NaN,NaN
1,1,1,2010-02-12,46039.49,24924.50,NaN,NaN
2,1,1,2010-02-19,41595.55,46039.49,NaN,NaN
3,1,1,2010-02-26,19403.54,41595.55,NaN,NaN
4,1,1,2010-03-05,21827.90,19403.54,24924.50,NaN
5,1,1,2010-03-12,21043.39,21827.90,46039.49,NaN
6,1,1,2010-03-19,22136.64,21043.39,41595.55,NaN
7,1,1,2010-03-26,26229.21,22136.64,19403.54,NaN
8,1,1,2010-04-02,57258.43,26229.21,21827.90,NaN
9,1,1,2010-04-09,42960.91,57258.43,21043.39,NaN


In [14]:
# save dataset before dropping lag missing rows
sales_df.to_csv('../data/sales_with_lag_features.csv', index=False)

In [15]:
# inspect missing values in lag features
sales_df[['Lag_1', 'Lag_4', 'Lag_12']].isnull().sum()

Lag_1      3331
Lag_4     13134
Lag_12    38615
dtype: int64

In [16]:
# remove lag_12 feature
sales_df = sales_df.drop(columns=['Lag_12'])

In [17]:
# drop rows missing lag_4
sales_df = sales_df.dropna(subset=['Lag_4'])

In [18]:
sales_df.isnull().sum()

Store           0
Dept            0
Date            0
Weekly_Sales    0
IsHoliday       0
Temperature     0
Fuel_Price      0
MarkDown1       0
MarkDown2       0
MarkDown3       0
MarkDown4       0
MarkDown5       0
CPI             0
Unemployment    0
Size            0
Month           0
Year            0
Quarter         0
WeekOfYear      0
Type_B          0
Type_C          0
Lag_1           0
Lag_4           0
dtype: int64

## Lag Feature Engineering

Lag features were created to help the forecasting models learn from previous sales behavior.

The following lag features were generated within each `Store` and `Dept` group:

* `Lag_1` → previous week's sales
* `Lag_4` → sales from four weeks earlier

These features help capture short-term sales trends, temporal dependencies, and recurring demand patterns in the retail data.

Rows with missing lag values were removed because the earlier periods did not contain enough historical sales information to generate complete lag features.


In [19]:
# create rolling median feature
sales_df['Rolling_Median_4'] = sales_df.groupby(['Store', 'Dept'])['Weekly_Sales'].transform(lambda x: x.shift(1).rolling(window=4).median())

In [20]:
# create rolling standard deviation feature
sales_df['Rolling_Std_4'] = sales_df.groupby(['Store', 'Dept'])['Weekly_Sales'].transform(lambda x: x.shift(1).rolling(window=4).std())

In [21]:
# inspect missing values in rolling features
sales_df[['Rolling_Median_4', 'Rolling_Std_4']].isnull().sum()

Rolling_Median_4    12832
Rolling_Std_4       12832
dtype: int64

## Rolling Window Feature Engineering

Rolling window features were created to help the forecasting models capture recent sales trends and short-term sales variability.

The following rolling features were generated within each `Store` and `Dept` group:

* `Rolling_Median_4` → median sales over the previous 4 weeks
* `Rolling_Std_4` → sales variability over the previous 4 weeks

A rolling median was used instead of a rolling mean because the sales data contains skewness and extreme sales spikes. The median provides a more robust representation of recent sales behavior.

The rolling standard deviation feature was created to help the models identify periods of stable and volatile sales activity.


In [22]:
# cyclical encoding for month
sales_df['Month_Sin'] = np.sin(2 * np.pi * sales_df['Month'] / 12)
sales_df['Month_Cos'] = np.cos(2 * np.pi * sales_df['Month'] / 12)

In [23]:
# cyclical encoding for week of year
sales_df['WeekOfYear_Sin'] = np.sin(2 * np.pi * sales_df['WeekOfYear'] / 52)
sales_df['WeekOfYear_Cos'] = np.cos(2 * np.pi * sales_df['WeekOfYear'] / 52)

In [24]:
sales_df[
    [
        'Month',
        'Month_Sin',
        'Month_Cos',
        'WeekOfYear',
        'WeekOfYear_Sin',
        'WeekOfYear_Cos'
    ]
].head()

,Month,Month_Sin,Month_Cos,WeekOfYear,WeekOfYear_Sin,WeekOfYear_Cos
4,3,1.000000,6.123234e-17,9,0.885456,4.647232e-01
5,3,1.000000,6.123234e-17,10,0.935016,3.546049e-01
6,3,1.000000,6.123234e-17,11,0.970942,2.393157e-01
7,3,1.000000,6.123234e-17,12,0.992709,1.205367e-01
8,4,0.866025,-5.000000e-01,13,1.000000,-1.608123e-16


## Cyclical Time Encoding

Cyclical encoding was applied to the `Month` and `WeekOfYear` features using sine and cosine transformations.

This was done to preserve the cyclical nature of time-based variables. For example, December and January are consecutive periods in reality, even though their numerical values (`12` and `1`) appear far apart.

The cyclical features created include:

* `Month_Sin`
* `Month_Cos`
* `WeekOfYear_Sin`
* `WeekOfYear_Cos`

These features help the forecasting models better capture repeating seasonal patterns and smooth transitions across time periods.


In [25]:
#inspect remaining missing values
sales_df.isnull().sum()

Store                   0
Dept                    0
Date                    0
Weekly_Sales            0
IsHoliday               0
Temperature             0
Fuel_Price              0
MarkDown1               0
MarkDown2               0
MarkDown3               0
MarkDown4               0
MarkDown5               0
CPI                     0
Unemployment            0
Size                    0
Month                   0
Year                    0
Quarter                 0
WeekOfYear              0
Type_B                  0
Type_C                  0
Lag_1                   0
Lag_4                   0
Rolling_Median_4    12832
Rolling_Std_4       12832
Month_Sin               0
Month_Cos               0
WeekOfYear_Sin          0
WeekOfYear_Cos          0
dtype: int64

In [26]:
# drop remaining rows with missing values (if any)
sales_df = sales_df.dropna()

In [27]:
# inspect final dataset shape
sales_df.shape

(395604, 29)

## Final Missing Value Handling

Additional missing values were generated during lag and rolling window feature creation because some early time periods did not contain enough historical observations.

These remaining missing values were removed after all forecasting features had been created to ensure that the final dataset contained complete and consistent historical information for modeling.

The final processed dataset contains 395,604 rows and 29 columns and is fully prepared for forecasting model development.


In [28]:
# sort data chronologically by date
sales_df = sales_df.sort_values(by='Date')

In [29]:
# create training dataset
train_df = sales_df[sales_df['Year'] < 2012]

# create testing dataset
test_df = sales_df[sales_df['Year'] >= 2012]

In [30]:
# inspect train and test shapes
print(f'Training set shape: {train_df.shape}')
print(f'\nTesting set shape: {test_df.shape}')

Training set shape: (268488, 29)

Testing set shape: (127116, 29)


In [31]:
# inspect training period
print(f'Training period: {train_df["Date"].min()} to {train_df["Date"].max()}')

# inspect testing period
print(f'Testing period: {test_df["Date"].min()} to {test_df["Date"].max()}')

Training period: 2010-04-02 00:00:00 to 2011-12-30 00:00:00
Testing period: 2012-01-06 00:00:00 to 2012-10-26 00:00:00


## Chronological Train-Test Split

The dataset was split chronologically to preserve the natural time order of the sales data and prevent data leakage.

The training dataset contains historical sales records from 2010 to 2011, while the testing dataset contains future sales observations from 2012.

This approach simulates real-world forecasting conditions where models are trained on past data and evaluated on unseen future periods.

The final split produced:

* 268,488 training rows
* 127,116 testing rows

Using a chronological split is important in forecasting because random splitting can introduce future information into the training process and lead to unrealistic model performance.


In [32]:
# define feature columns
feature_cols = sales_df.drop(columns=['Weekly_Sales', 'Date']).columns.tolist()
feature_cols
    

['Store',
 'Dept',
 'IsHoliday',
 'Temperature',
 'Fuel_Price',
 'MarkDown1',
 'MarkDown2',
 'MarkDown3',
 'MarkDown4',
 'MarkDown5',
 'CPI',
 'Unemployment',
 'Size',
 'Month',
 'Year',
 'Quarter',
 'WeekOfYear',
 'Type_B',
 'Type_C',
 'Lag_1',
 'Lag_4',
 'Rolling_Median_4',
 'Rolling_Std_4',
 'Month_Sin',
 'Month_Cos',
 'WeekOfYear_Sin',
 'WeekOfYear_Cos']

In [33]:
# create training features
X_train = train_df[feature_cols]

# create training target
y_train = train_df['Weekly_Sales']

# create testing features
X_test = test_df[feature_cols]

# create testing target
y_test = test_df['Weekly_Sales']

In [34]:
# inspect feature and target shapes
print(f'X_train shape: {X_train.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'\nX_test shape: {X_test.shape}')
print(f'y_test shape: {y_test.shape}')

X_train shape: (268488, 27)
y_train shape: (268488,)

X_test shape: (127116, 27)
y_test shape: (127116,)


In [35]:
# save unscaled feature-engineered dataset
sales_df.to_csv('../data/feature_engineered_sales.csv', index=False)

## Feature and Target Separation

The dataset was separated into predictor variables (`X`) and the target variable (`y`) for machine learning modeling.

`Weekly_Sales` was used as the target variable, while the remaining engineered and processed features were used as predictors.

The `Date` column was excluded from the modeling dataset because relevant time-based information had already been extracted through feature engineering.

The final datasets created include:

* `X_train`
* `y_train`
* `X_test`
* `y_test`

These datasets will be used for forecasting model training and evaluation.


In [36]:
# define columns to scale
scale_cols = [
    'Temperature',
    'Fuel_Price',
    'MarkDown1',
    'MarkDown2',
    'MarkDown3',
    'MarkDown4',
    'MarkDown5',
    'CPI',
    'Unemployment',
    'Size',
    'Lag_1',
    'Lag_4',
    'Rolling_Median_4',
    'Rolling_Std_4'
]

In [37]:
from sklearn.preprocessing import RobustScaler

# initialize scaler
scaler = RobustScaler()

# Create a copy of the training and testing features to scale
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Fit the scaler on the training data and transform both training and testing data
X_train_scaled[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test_scaled[scale_cols] = scaler.transform(X_test[scale_cols])

In [38]:
X_train_scaled.head()

,Store,Dept,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Size,Month,Year,Quarter,WeekOfYear,Type_B,Type_C,Lag_1,Lag_4,Rolling_Median_4,Rolling_Std_4,Month_Sin,Month_Cos,WeekOfYear_Sin,WeekOfYear_Cos
8,1,1,0,-0.017651,-0.634271,0.0,0.0,0.0,0.0,0.0,0.360548,-0.128793,0.102400,4,2010,2,13,0,0,1.005488,0.778219,0.794431,0.938353,0.866025,-0.5,1.0,-1.608123e-16
340999,36,7,0,0.024135,-0.648338,0.0,0.0,0.0,0.0,0.0,0.345014,0.356773,-0.920913,4,2010,2,13,0,0,-0.413093,-0.411088,-0.414486,-0.433803,0.866025,-0.5,1.0,-1.608123e-16
341142,36,8,0,0.024135,-0.648338,0.0,0.0,0.0,0.0,0.0,0.345014,0.356773,-0.920913,4,2010,2,13,0,0,-0.235882,-0.239876,-0.238875,-0.245462,0.866025,-0.5,1.0,-1.608123e-16
65802,7,60,0,-0.882565,-0.626598,0.0,0.0,0.0,0.0,0.0,0.093132,0.726129,-0.637971,4,2010,2,13,1,0,-0.423625,-0.423698,-0.426990,-0.450313,0.866025,-0.5,1.0,-1.608123e-16
65685,7,59,0,-0.882565,-0.626598,0.0,0.0,0.0,0.0,0.0,0.093132,0.726129,-0.637971,4,2010,2,13,1,0,-0.417166,-0.418507,-0.419664,-0.440951,0.866025,-0.5,1.0,-1.608123e-16


In [39]:
X_test_scaled.head()

,Store,Dept,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Size,Month,Year,Quarter,WeekOfYear,Type_B,Type_C,Lag_1,Lag_4,Rolling_Median_4,Rolling_Std_4,Month_Sin,Month_Cos,WeekOfYear_Sin,WeekOfYear_Cos
417161,45,40,0,-0.968300,0.264706,7328.14,33378.79,34.60,1198.48,6819.10,0.086663,0.327165,-0.201585,1,2012,1,1,1,0,1.675419,1.581886,1.761815,1.900929,0.5,0.866025,0.120537,0.992709
208518,22,10,0,-1.014049,0.286445,5803.25,34453.80,22.94,1777.64,8239.01,-0.528166,-0.354552,-0.189314,1,2012,1,1,1,0,0.526259,0.665967,0.703575,1.636228,0.5,0.866025,0.120537,0.992709
160230,17,21,0,-1.265130,-0.172634,5185.45,12090.50,33.90,1490.06,4832.68,-0.660997,-1.168764,-0.431527,1,2012,1,1,1,0,-0.242637,-0.160075,-0.123299,0.769320,0.5,0.866025,0.120537,0.992709
313965,32,98,0,-0.936960,-0.153453,4323.27,33209.84,208.29,1848.90,6197.53,0.179769,0.202813,0.577218,1,2012,1,1,0,0,-0.050848,0.120797,0.155087,1.045920,0.5,0.866025,0.120537,0.992709
169001,18,12,0,-1.134366,0.286445,6196.77,43731.43,128.45,1653.17,8527.20,-0.578165,0.068838,-0.179246,1,2012,1,1,1,0,-0.304541,-0.264471,-0.278118,-0.206565,0.5,0.866025,0.120537,0.992709


In [40]:
from sklearn.preprocessing import PowerTransformer

# initialize 
pt = PowerTransformer(method='yeo-johnson')

# Fit and transform on target
y_train_log = pt.fit_transform(y_train.values.reshape(-1, 1)).squeeze()
print(y_train_log.shape)

y_test_log = pt.transform(y_test.values.reshape(-1, 1)).squeeze()
print(y_test_log.shape)

(268488,)
(127116,)


In [46]:
import joblib

# save fitted transformer
joblib.dump( pt, '../model/power_transformer.pkl')

['../model/power_transformer.pkl']

In [ ]:
# save scaled features and transformed target
X_train_scaled.to_csv('../data/X_train_scaled.csv', index=False)
X_test_scaled.to_csv('../data/X_test_scaled.csv', index=False)
pd.DataFrame(y_train_log, columns=['Weekly_Sales_Log']).to_csv('../data/y_train_log.csv', index=False)
pd.DataFrame(y_test_log, columns=['Weekly_Sales_Log']).to_csv('../data/y_test_log.csv', index=False)    
X_train.to_csv('../data/X_train_unscaled.csv', index=False)
X_test.to_csv('../data/X_test_unscaled.csv', index=False)
pd.DataFrame(y_train, columns=['Weekly_Sales']).to_csv('../data/y_train_unscaled.csv', index=False)
pd.DataFrame(y_test, columns=['Weekly_Sales']).to_csv('../data/y_test_unscaled.csv', index=False)

## Forecasting Dataset Preparation and Transformation

The final forecasting dataset was prepared after completing feature engineering, lag creation, rolling window features, and cyclical encoding.

Remaining missing values generated during lag and rolling feature creation were removed to ensure that all observations contained complete historical information for forecasting.

The dataset was sorted chronologically and split into training and testing datasets based on time order to prevent data leakage.

* Training data: 2010–2011
* Testing data: 2012

The target variable used for prediction is `Weekly_Sales`, while the remaining engineered features were used as predictor variables.

Feature scaling was applied to the predictor variables to standardize feature magnitudes for scale-sensitive machine learning models.

A Yeo-Johnson transformation was applied to the target variable to reduce skewness and stabilize the sales distribution before modeling.

The final datasets created include:

* `X_train`
* `X_test`
* `X_train_scaled`
* `X_test_scaled`
* `y_train`
* `y_test`
* `y_train_log`
* `y_test_log`

The processed forecasting dataset contains:

* 395,604 rows
* 29 columns

and is fully prepared for machine learning model development.
